# USMA1Q — Méthodes Numériques
## Exercices Séance 4 — Tableaux en mémoire, matrices & précision numérique

## Partie 1 — Vues et copies

### Exercice 1.1 — Tableau d'éprouvettes

On dispose de données pour 10 éprouvettes d'acier. Chaque ligne correspond à une éprouvette, les colonnes sont :  
- colonne 0 : résistance à la traction (MPa)  
- colonne 1 : allongement à la rupture (%)  
- colonne 2 : dureté (HV)

1. Créez le tableau `data` à partir des valeurs ci-dessous.  
2. Extrayez la colonne de dureté dans une variable `durete`. Vérifiez avec `np.shares_memory()` que c'est bien une vue.  
3. Modifiez `durete[0] = 0`. Observez comment `data` a changé.  
4. Repartez du tableau original (recréez-le). Extrayez la colonne de dureté avec `.copy()`. Refaites la modification et vérifiez que `data` est intact.  
5. À l'aide de `%timeit`, comparez la vitesse de `np.sum(big, axis=0)` vs `np.sum(big, axis=1)` pour un grand tableau `big = np.random.rand(10000, 10000)`. Quel axe est plus rapide ? Pourquoi ?

In [ ]:
import numpy as np

# Données brutes : [UTS (MPa), allongement (%), dureté (HV)]
valeurs = [
    [510, 22, 152],
    [495, 24, 148],
    [522, 21, 156],
    [503, 23, 150],
    [515, 22, 154],
    [498, 25, 147],
    [519, 21, 155],
    [507, 23, 151],
    [501, 24, 149],
    [513, 22, 153],
]

# 1. Créer le tableau
data = np.array(valeurs)
print("data.shape :", data.shape)
print("data.dtype :", data.dtype)

In [ ]:
# 2. Extraire la colonne de dureté — est-ce une vue ?
durete = data[:, 2]
print("Partage mémoire data/durete :", np.shares_memory(data, durete))

# 3. Modifier durete[0] — observer l'effet sur data
# À compléter : modifiez durete[0] = 0, puis affichez data[0].
# Que constatez-vous ?


# 4. Repartez du tableau original, extrayez la colonne avec .copy(), refaites la modification.
# À compléter : recréez data, faites la copie, modifiez-la, vérifiez que data est intact.


In [ ]:
# 5. Benchmark axis=0 vs axis=1
# NB : ceci peut prendre quelques secondes
big = np.random.rand(5000, 5000)

# Décommentez et exécutez ces lignes si vous avez Jupyter :
# %timeit np.sum(big, axis=0)   # somme sur les lignes → résultat shape (5000,)
# %timeit np.sum(big, axis=1)   # somme sur les colonnes → résultat shape (5000,)

# Version portable sans %timeit :
import time

t0 = time.perf_counter()
for _ in range(10):
    np.sum(big, axis=0)
t1 = time.perf_counter()
print("axis=0 (10 répétitions) :", round(t1-t0, 3), "s")

t0 = time.perf_counter()
for _ in range(10):
    np.sum(big, axis=1)
t1 = time.perf_counter()
print("axis=1 (10 répétitions) :", round(t1-t0, 3), "s")

# À compléter : expliquez le résultat observé en termes d'ordre de stockage (C-order).
# Quel axe est plus rapide ? Pourquoi ?


## Partie 2 — Opérations matricielles

### Exercice 2.1 — Système de 3 ressorts en série

Trois ressorts de raideurs $k_1 = 100$ N/mm, $k_2 = 150$ N/mm, $k_3 = 80$ N/mm sont assemblés en série. En éléments finis 1D, la matrice de rigidité assemblée pour les degrés de liberté internes $(u_1, u_2, u_3)$ est :

$$K = \begin{pmatrix} k_1 & -k_1 & 0 \\ -k_1 & k_1+k_2 & -k_2 \\ 0 & -k_2 & k_2+k_3 \end{pmatrix}$$

1. Construisez `K` à la main avec `np.array()`.  
2. Construisez le même `K` de façon programmatique en utilisant `np.diag()` — en une ou deux lignes.  
3. Vérifiez que les deux matrices sont égales avec `np.allclose()`.  
4. Calculez le vecteur de déplacement $u$ si la force appliquée est $F = [0, 0, 500]$ N (les deux premiers nœuds sont libres, le troisième est soumis à la force) via `np.linalg.solve(K, F)` (*on approfondit cela en séance 5*).  
5. Calculez la force résultante `K @ u` et vérifiez qu'elle est bien égale à `F`.

In [ ]:
import numpy as np

k1, k2, k3 = 100, 150, 80

# 1. Construction directe
K_direct = np.array([
    [ k1,        -k1,        0      ],
    [-k1,         k1 + k2,  -k2     ],
    [  0,        -k2,        k2 + k3]
], dtype=float)
print("K =", K_direct)

In [ ]:
# 2. Construction programmatique avec np.diag()
# Rappel : np.diag(v) crée une matrice diagonale à partir d'un vecteur
#          np.diag(v, k=1) place v sur la sur-diagonale (k diagonals above main)
#          np.diag(v, k=-1) place v sur la sous-diagonale

# À compléter : construisez K_prog en utilisant np.diag() et des additions de matrices.
# Indice : vous avez besoin de 3 appels à np.diag().
K_prog = ...  # remplacez ...

# 3. Vérification
print("Matrices égales :", np.allclose(K_direct, K_prog))


In [ ]:
# 4. Résolution Ku = F
F = np.array([0.0, 0.0, 500.0])

# À compléter : résolvez le système K @ u = F avec np.linalg.solve()
u = ...  # remplacez ...
print("Vecteur déplacement u (mm) :", np.round(u, 4))

# 5. À compléter : calculez K @ u et vérifiez que le résultat est égal à F
# (utilisez np.allclose() pour la vérification)


### Exercice 2.2 — Opérations sur une matrice de mesures

Vous avez mesuré les modules de Young de 4 matériaux selon 3 directions différentes (résultats en GPa) :

|           | Direction 1 | Direction 2 | Direction 3 |
|-----------|-------------|-------------|-------------|
| Matériau A | 210 | 208 | 212 |
| Matériau B | 195 | 198 | 193 |
| Matériau C | 70  | 72  | 69  |
| Matériau D | 116 | 114 | 118 |

**Rappel — norme de Frobenius :** pour une matrice $M$ de taille $m \times n$, la norme de Frobenius est la racine de la somme des carrés de tous les éléments :
$$\|M\|_F = \sqrt{\sum_{i=1}^{m}\sum_{j=1}^{n} m_{ij}^2}$$
C'est l'extension naturelle de la norme euclidienne aux matrices. Dans NumPy : `np.linalg.norm(M)` (c'est la norme par défaut pour les matrices 2D).

1. Créez la matrice `E` (4 × 3).  
2. Calculez la moyenne par matériau (axis=1) et par direction (axis=0).  
3. Calculez la matrice de covariance entre matériaux via `np.cov(E)` (*chaque ligne est une variable, chaque colonne est une observation*).  
4. Calculez la norme de Frobenius de `E`.  
5. Calculez le produit $E^T E$ (matrice 3×3). Quelle est sa trace ? Comparez à la norme de Frobenius au carré.

In [ ]:
import numpy as np

E = np.array([
    [210, 208, 212],
    [195, 198, 193],
    [ 70,  72,  69],
    [116, 114, 118]
], dtype=float)

# 2. Moyennes
moy_materiaux = np.mean(E, axis=1)   # moyenne sur les colonnes (directions)
moy_directions = np.mean(E, axis=0)  # moyenne sur les lignes (matériaux)
print("Moyenne par matériau :", moy_materiaux, "GPa")
print("Moyenne par direction :", moy_directions, "GPa")

# 3. À compléter : calculez la matrice de covariance avec np.cov(E)
# Quelle est sa taille ? Que représente chaque coefficient ?


# 4. À compléter : calculez la norme de Frobenius de E avec np.linalg.norm()


# 5. À compléter : calculez E^T @ E, sa trace (np.trace()), et comparez à ||E||_F²
# Que constatez-vous ?


## Partie 3 — Précision et virgule flottante

### Exercice 3.1 — Epsilon machine et petites perturbations

1. Vérifiez que `0.1 + 0.2 == 0.3` renvoie `False`. Affichez `0.1 + 0.2` en pleine précision avec `repr()`.  
2. Affichez `np.finfo(np.float64).eps`. Quelle en est la signification ?  
3. Calculez `(1 + 1e-16) - 1`. Expliquez le résultat.  
4. Calculez `(1 + 1e-8) - 1`. Est-ce exact ? Quelle est l'erreur relative ?  
5. Calculez `(1 + 1e-8) - 1` en utilisant `np.float128`. La précision s'améliore-t-elle ?

In [ ]:
import numpy as np

# 1. 0.1 + 0.2
print("0.1 + 0.2 == 0.3 :", 0.1 + 0.2 == 0.3)
print("repr(0.1 + 0.2)  :", repr(0.1 + 0.2))

# 2. À compléter : affichez np.finfo(np.float64).eps
# Expliquez (en commentaire ou print) la signification de cette valeur.


In [ ]:
# 3. À compléter : calculez (1 + 1e-16) - 1. Expliquez pourquoi le résultat est 0.0.


# 4. À compléter : calculez (1 + 1e-8) - 1.
# Calculez l'erreur relative par rapport à la valeur exacte (1e-8).


# 5. À compléter : refaites le calcul de la question 4 en np.float128. La précision s'améliore-t-elle ?


### Exercice 3.2 — Somme de la série $\sum_{n=1}^{N} \frac{1}{n^2}$

La série $\sum_{n=1}^{\infty} \frac{1}{n^2}$ converge vers $\pi^2/6$ (résultat de Bâle).

1. Pour $N = 10\,000$, calculez la somme **croissante** (de $n=1$ à $N$) et la somme **décroissante** (de $n=N$ à $1$). Comparez les deux résultats à $\pi^2/6$.  
2. Quelle méthode est plus précise ? Pourquoi ?  
3. *(Bonus)* Essayez avec `np.float32` et comparez les erreurs.

In [ ]:
import numpy as np

N = 10000
vrai = np.pi**2 / 6

# 1. À compléter : calculez la somme croissante (n=1 à N) et décroissante (n=N à 1)
# Indice : utilisez np.arange() et np.sum()
n_croissant = np.arange(1, N + 1)
somme_croissante = ...   # remplacez ...


somme_decroissante = ... # remplacez ... (pensez à inverser l'ordre)


print("Valeur vraie (π²/6)     :", vrai)
print("Somme croissante (1→N)  :", somme_croissante, "  erreur :", abs(somme_croissante - vrai))
print("Somme décroissante (N→1):", somme_decroissante, "  erreur :", abs(somme_decroissante - vrai))

# 2. À compléter : expliquez (en commentaire ou print) quelle méthode est plus précise
# et pourquoi.


In [ ]:
# 3. Bonus : comparaison float32 vs float64
# À compléter : recalculez la somme en float32 et comparez l'erreur à celle en float64.
# Indice : utilisez n.astype(np.float32) et np.float32(1.0) pour forcer le type.


## Partie 4 — Propagation d'incertitude

### Exercice 4.1 — Contrainte avec incertitude

Cinq éprouvettes cylindriques ont les mesures suivantes :

| Éprouvette | Force rupture (N) | Diamètre (mm) |
|------------|-------------------|----------------|
| 1 | 45 200 | 10.02 |
| 2 | 43 800 | 9.98  |
| 3 | 46 100 | 10.05 |
| 4 | 44 500 | 10.00 |
| 5 | 45 700 | 10.03 |

L'incertitude instrumentale est $\delta F = 100$ N et $\delta d = 0{,}02$ mm.

La contrainte d'ingénierie est $\sigma = F / A = 4F / (\pi d^2)$ et son incertitude :

$$\delta\sigma = \sigma \sqrt{\left(\frac{\delta F}{F}\right)^2 + 4\left(\frac{\delta d}{d}\right)^2}$$

1. **Sans boucle** (`for`), calculez vectoriellement $\sigma$ et $\delta\sigma$ pour les 5 éprouvettes.  
2. Affichez les résultats sous forme de tableau : `"σ = xxx.x ± y.y MPa"`.  
3. Quelle est la contrainte moyenne ? Quel est l'écart-type des contraintes mesurées ?  
4. Pour quelle éprouvette l'incertitude instrumentale est-elle relativement la plus grande ?

In [ ]:
import numpy as np

forces = np.array([45200, 43800, 46100, 44500, 45700], dtype=float)  # N
diametres = np.array([10.02, 9.98, 10.05, 10.00, 10.03])             # mm

delta_F = 100.0   # N
delta_d = 0.02    # mm

# 1. À compléter : calculez vectoriellement sigma et delta_sigma (sans boucle for).
# Rappel : A = pi * (d/2)², sigma = F / A
aires = ...       # remplacez ...
sigma = ...       # remplacez ...
delta_sigma = ... # remplacez ... (formule de propagation donnée dans l'énoncé)

# 2. Affichage formaté (cette boucle sert uniquement à afficher, pas à calculer)
for i in range(len(forces)):
    print("Eprouvette ", i + 1, ": sigma = ", round(sigma[i], 1), " +/- ", round(delta_sigma[i], 1), " MPa")

# 3. À compléter : calculez la contrainte moyenne et l'écart-type


# 4. À compléter : trouvez l'éprouvette avec l'incertitude relative la plus grande
# Indice : incert_rel = delta_sigma / sigma, puis np.argmax()


### Exercice 4.2 — Propagation sur un volume

Une pièce cylindrique creuse a les dimensions nominales :  
- diamètre extérieur $D = 50$ mm ± 0,05 mm  
- diamètre intérieur $d = 40$ mm ± 0,05 mm  
- longueur $L = 200$ mm ± 0,1 mm

Le volume est $V = \frac{\pi}{4}(D^2 - d^2) L$.

1. Calculez le volume nominal.  
2. Dérivez analytiquement $\delta V / V$ en fonction de $\delta D$, $\delta d$, $\delta L$.  
3. Calculez $\delta V$ numériquement.  
4. **Bonus / vérification :** estimez $\delta V$ par différences finies : $V(D + \delta D, d, L) - V(D, d, L)$, etc.  
5. Quelle source d'incertitude (D, d ou L) contribue le plus à $\delta V$ ? Justifiez.

In [ ]:
import numpy as np

D, delta_D = 50.0, 0.05    # mm
d, delta_d = 40.0, 0.05    # mm
L, delta_L = 200.0, 0.1    # mm

# 1. Calculez le volume nominal
V = np.pi / 4 * (D**2 - d**2) * L
print("Volume nominal V =", round(V, 2), "mm^3")

# 2 & 3. À compléter : dérivez analytiquement les dérivées partielles, puis calculez delta_V par la formule de propagation.
# Rappel : dV/dD = pi*D*L/2,  dV/dd = -pi*d*L/2,  dV/dL = pi/4*(D²-d²)
delta_V = ...  # remplacez ...

print("delta_V =", round(delta_V, 2), "mm^3")
print("delta_V/V =", round(delta_V/V * 100, 4), "%")

# 4. Bonus : À compléter — estimez delta_V par différences finies.
# Calculez V(D+δD, d, L) - V(D, d, L), etc., puis combinez.


# 5. À compléter : quelle source d'incertitude (D, d ou L) contribue le plus ?
# Calculez la contribution relative de chaque terme.
